In [2]:
import os
import urllib
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

In [3]:
#Load raw dataset
raw_csv_path = '../data/healthcare-dataset-stroke-data.csv'
df = pd.read_csv(raw_csv_path)

In [4]:
# Output the column names of the Pandas dataframe
df.columns

Index(['id', 'gender', 'age', 'hypertension', 'heart_disease', 'ever_married',
       'work_type', 'Residence_type', 'avg_glucose_level', 'bmi',
       'smoking_status', 'stroke'],
      dtype='str')

In [5]:
#Standardize column names to snake_case
df.columns = df.columns.str.lower()

In [13]:
#Filter out single 'Other' gender row
df = df[df['gender']!='Other'].copy()

In [14]:
#Bin age into demographic groups
bins=[0,18,35,50,65,120]
labels = ['0-18', '19-35', '36-50', '51-65', '65+']
df['age_group'] = pd.cut(df['age'],bins=bins,labels=labels,right=True)

In [15]:
#Impute missing BMI values using median by (age_group, gender)
df['bmi'] = df.groupby(['age_group','gender'],observed=False)['bmi'].transform(
    lambda x:x.fillna(x.median())
)
df['bmi'] = df['bmi'].round(2)
df['avg_glucose_level'] = df['avg_glucose_level'].round(2)

In [17]:
# Classify BMI values into standard weight categories using a custom function
def classify_bmi(bmi):
    if bmi<18.5:
        return 'Underweight'
    elif 18.5<=bmi<25:
        return 'Normal Weight'
    elif 25<=bmi<30:
        return 'Overweight'
    else:
         return 'Obese'
df['bmi_category'] = df['bmi'].apply(classify_bmi)

In [18]:
#Map binary indicators to clean string labels for filters
df['hypertension_label'] = df['hypertension'].map({1:'Yes',0:'No'})
df['heart_disease_label'] = df['heart_disease'].map({1:'Yes',0:'No'})
df['stroke_label'] = df['stroke'].map({1:'Stroke',0:'No Stroke'})

In [19]:
# Convert categorical columns to string for database compatibility
df['age_group'] = df['age_group'].astype(str)

In [20]:
#Save cleaned CSV
df.to_csv('../data/cleaned_stroke_data.csv',index=False)
print('Cleaned CSV saved successfully.')

Cleaned CSV saved successfully.


In [21]:
#Ingest into SQL Server
server = 'localhost'
database = 'Healthcare_Analytics'

In [22]:
# Construct SQLAlchemy connection string for SQL Server using Windows authentication
connection_string = (
    f"mssql+pyodbc://{server}/{database}"
    "?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)

In [23]:
# Create a SQLAlchemy engine instance using the constructed connection string
engine = create_engine(connection_string)

In [24]:
# Load into table 'stroke_data'
df.to_sql('stroke_data',con=engine,if_exists='replace',index=False)
print("Data successfully ingested into SQL Server table 'stroke_data'.")

C:\Users\Hassa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pandas\io\sql.py:1649: SAWarning: Unrecognized server version info '17.0.1135.8'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


Data successfully ingested into SQL Server table 'stroke_data'.
